# Extração SIGTAP — Tabela de Procedimentos, Medicamentos e OPM do SUS para o projeto Conecta Saúde

Este notebook documenta a fonte **SIGTAP — Sistema de Gerenciamento da Tabela de Procedimentos, Medicamentos e OPM do SUS**.

O objetivo é mostrar:

1. **quais dados serão extraídos**;
2. **qual endpoint público será usado**;
3. **como realizar a extração com Python**;
4. **quais campos são úteis para o projeto**;
5. **quais perguntas de negócio podem ser respondidas com esses dados**.

O SIGTAP será usado como **dicionário e camada semântica** dos procedimentos. Ele transforma códigos técnicos do SIH/SUS e SIA/SUS em nomes, grupos, subgrupos, formas de organização, complexidade e atributos úteis para análise.


## 1. Papel do SIGTAP no projeto

Dentro do projeto, o SIGTAP será usado para enriquecer:

- `PROC_REA` e `PROC_SOLIC` do SIH/SUS;
- `PA_PROC_ID` do SIA/SUS;
- rankings de procedimentos;
- análise por grupo e subgrupo assistencial;
- perguntas em linguagem natural com Select AI;
- storytelling executivo no dashboard.

Sem o SIGTAP, o painel exibiria códigos de procedimentos. Com o SIGTAP, é possível mostrar nomes e categorias compreensíveis para gestores.


## 2. Endpoint público de conexão

A página pública de download do SIGTAP é:

```text
https://sigtap.datasus.gov.br/tabela-unificada/app/download.jsp
```

Espelho frequentemente usado:

```text
https://tabela-unificada.datasus.gov.br/tabela-unificada/app/download.jsp
```

A área de download disponibiliza, por competência/mês, arquivos compactados contendo tabelas `.txt` da Tabela de Procedimentos do SUS.

> Observação: diferente do SIH/SUS, SIA/SUS e CNES, o SIGTAP costuma exigir leitura de arquivos `.txt` de largura fixa dentro de um `.zip`. Os layouts podem mudar ao longo do tempo. Por isso, este notebook implementa um parser mínimo e seguro para a tabela de procedimentos, suficiente para enriquecer os códigos do MVP.


In [ ]:
# Instalação das dependências
# Execute esta célula no Colab/Jupyter caso os pacotes ainda não estejam instalados.

%pip install -q pandas pyarrow requests beautifulsoup4


## 3. Parâmetros iniciais


In [ ]:
from pathlib import Path
from urllib.parse import urljoin
from zipfile import ZipFile
import re
import pandas as pd
import requests
from bs4 import BeautifulSoup

URL_DOWNLOAD_SIGTAP = "https://sigtap.datasus.gov.br/tabela-unificada/app/download.jsp"
URL_DOWNLOAD_SIGTAP_ESPELHO = "https://tabela-unificada.datasus.gov.br/tabela-unificada/app/download.jsp"

DIRETORIO_RAW = Path("dados/raw/sigtap")
DIRETORIO_TRATADO = Path("dados/tratado/sigtap")

DIRETORIO_RAW.mkdir(parents=True, exist_ok=True)
DIRETORIO_TRATADO.mkdir(parents=True, exist_ok=True)


## 4. Listagem de links de download disponíveis

A função abaixo tenta encontrar links de arquivos `.zip` na página de download.

Caso o site esteja com download protegido por sessão, JavaScript ou redirecionamento, a alternativa prática é baixar manualmente o `.zip` pelo navegador e colocar o arquivo em `dados/raw/sigtap/`.


In [ ]:
def listar_links_download_sigtap(url: str = URL_DOWNLOAD_SIGTAP) -> pd.DataFrame:
    resposta = requests.get(url, timeout=60)
    resposta.raise_for_status()

    soup = BeautifulSoup(resposta.text, "html.parser")
    linhas = []

    for a in soup.find_all("a"):
        href = a.get("href")
        texto = " ".join(a.get_text(" ", strip=True).split())
        if not href:
            continue

        link_absoluto = urljoin(url, href)
        eh_candidato = any(p in link_absoluto.lower() for p in [".zip", "download", "arquivo", "competencia"])

        if eh_candidato:
            linhas.append({
                "texto": texto,
                "href": href,
                "url": link_absoluto,
            })

    return pd.DataFrame(linhas)

links_sigtap = listar_links_download_sigtap(URL_DOWNLOAD_SIGTAP)
links_sigtap.head(20)


## 5. Download do arquivo ZIP

Se `links_sigtap` retornar um link direto para `.zip`, informe a URL em `URL_ZIP_SIGTAP`.

Se não retornar, faça o download manual no site do SIGTAP e coloque o arquivo ZIP em `dados/raw/sigtap/`. Depois defina `ARQUIVO_ZIP_LOCAL` com o caminho do arquivo.


In [ ]:
# Opção A: download automático por URL direta, quando disponível.
URL_ZIP_SIGTAP = None  # Ex.: "https://.../TabelaUnificada_202401.zip"

# Opção B: arquivo baixado manualmente pelo navegador.
ARQUIVO_ZIP_LOCAL = None  # Ex.: "dados/raw/sigtap/TabelaUnificada_202401.zip"


def baixar_zip_sigtap(url_zip: str, diretorio_destino: Path = DIRETORIO_RAW) -> Path:
    nome = url_zip.split("/")[-1].split("?")[0] or "sigtap.zip"
    destino = diretorio_destino / nome

    resposta = requests.get(url_zip, timeout=120)
    resposta.raise_for_status()

    destino.write_bytes(resposta.content)
    return destino


if URL_ZIP_SIGTAP:
    caminho_zip = baixar_zip_sigtap(URL_ZIP_SIGTAP)
elif ARQUIVO_ZIP_LOCAL:
    caminho_zip = Path(ARQUIVO_ZIP_LOCAL)
else:
    # Procura automaticamente um ZIP já colocado na pasta raw.
    arquivos_zip = list(DIRETORIO_RAW.glob("*.zip"))
    caminho_zip = arquivos_zip[0] if arquivos_zip else None

print("Arquivo ZIP selecionado:", caminho_zip)


## 6. Inspeção dos arquivos dentro do ZIP

O pacote SIGTAP costuma trazer vários arquivos `.txt`, por exemplo:

- tabela de procedimentos;
- grupos;
- subgrupos;
- formas de organização;
- relacionamento com CID;
- relacionamento com CBO;
- compatibilidades;
- serviços/classificações;
- financiamento;
- atributos complementares.

Para o MVP, a tabela mais importante é a de procedimentos, geralmente com nome semelhante a `tb_procedimento.txt`.


In [ ]:
def listar_arquivos_zip(caminho_zip: Path) -> pd.DataFrame:
    if caminho_zip is None or not Path(caminho_zip).exists():
        raise FileNotFoundError(
            "Nenhum ZIP do SIGTAP encontrado. Baixe manualmente pelo site e coloque em dados/raw/sigtap/."
        )

    with ZipFile(caminho_zip) as zf:
        linhas = []
        for info in zf.infolist():
            linhas.append({
                "arquivo": info.filename,
                "tamanho_bytes": info.file_size,
            })
    return pd.DataFrame(linhas)

if caminho_zip:
    arquivos_no_zip = listar_arquivos_zip(caminho_zip)
    display(arquivos_no_zip.head(50))
else:
    print("Baixe o ZIP do SIGTAP e execute novamente a célula.")


## 7. Leitura da tabela de procedimentos

A função abaixo procura um arquivo de procedimentos dentro do ZIP e extrai um conjunto mínimo de campos:

| Campo extraído | Uso no projeto |
|---|---|
| `co_procedimento` | Chave para cruzar com SIH/SUS e SIA/SUS |
| `no_procedimento` | Nome do procedimento para dashboard e Select AI |
| `co_grupo` | Grupo derivado dos dois primeiros dígitos |
| `co_subgrupo` | Subgrupo derivado do 3º e 4º dígitos |
| `co_forma_organizacao` | Forma de organização derivada do 5º e 6º dígitos |

> Este parser mínimo é propositalmente conservador. Para produção final, o ideal é ler também o layout oficial da competência e expandir os campos de valor, complexidade, financiamento, CBO, CID, habilitação e compatibilidades.


In [ ]:
def encontrar_arquivo_procedimento(caminho_zip: Path) -> str:
    with ZipFile(caminho_zip) as zf:
        nomes = zf.namelist()

    candidatos = [
        n for n in nomes
        if "proced" in n.lower() and n.lower().endswith(".txt")
    ]

    if not candidatos:
        raise FileNotFoundError("Arquivo de procedimentos não encontrado no ZIP.")

    # Prioriza tb_procedimento quando existir.
    candidatos_ordenados = sorted(candidatos, key=lambda x: ("tb_procedimento" not in x.lower(), x))
    return candidatos_ordenados[0]


def ler_tb_procedimento_minimo(caminho_zip: Path, encoding: str = "latin-1") -> pd.DataFrame:
    arquivo_proc = encontrar_arquivo_procedimento(caminho_zip)

    with ZipFile(caminho_zip) as zf:
        conteudo = zf.read(arquivo_proc).decode(encoding, errors="replace").splitlines()

    linhas = []
    for linha in conteudo:
        linha = linha.rstrip("\n")
        if not linha.strip():
            continue

        # Código do procedimento no SIGTAP possui 10 dígitos.
        co_procedimento = linha[0:10].strip()
        no_procedimento = linha[10:260].strip()

        if not re.fullmatch(r"\d{10}", co_procedimento):
            continue

        linhas.append({
            "co_procedimento": co_procedimento,
            "no_procedimento": no_procedimento,
            "co_grupo": co_procedimento[0:2],
            "co_subgrupo": co_procedimento[2:4],
            "co_forma_organizacao": co_procedimento[4:6],
            "arquivo_origem": arquivo_proc,
        })

    return pd.DataFrame(linhas)

if caminho_zip:
    df_procedimentos = ler_tb_procedimento_minimo(caminho_zip)
    print(df_procedimentos.shape)
    display(df_procedimentos.head())
else:
    df_procedimentos = pd.DataFrame()
    print("Baixe o ZIP do SIGTAP e execute novamente a célula.")


## 8. Classificação textual dos grupos principais

Os dois primeiros dígitos do procedimento indicam o grupo. Esta classificação já melhora muito o dashboard.


In [ ]:
MAPA_GRUPOS_SIGTAP = {
    "01": "Ações de promoção e prevenção em saúde",
    "02": "Procedimentos com finalidade diagnóstica",
    "03": "Procedimentos clínicos",
    "04": "Procedimentos cirúrgicos",
    "05": "Transplantes de órgãos, tecidos e células",
    "06": "Medicamentos",
    "07": "Órteses, próteses e materiais especiais",
    "08": "Ações complementares da atenção à saúde",
}

if not df_procedimentos.empty:
    df_procedimentos["grupo_procedimento"] = df_procedimentos["co_grupo"].map(MAPA_GRUPOS_SIGTAP)
    display(df_procedimentos.head())


## 9. Quais dados serão extraídos

| Campo | Uso no projeto |
|---|---|
| `co_procedimento` | Chave para cruzar com SIH/SUS (`PROC_REA`) e SIA/SUS (`PA_PROC_ID`) |
| `no_procedimento` | Nome legível do procedimento |
| `co_grupo` | Grupo do procedimento |
| `grupo_procedimento` | Nome do grupo assistencial |
| `co_subgrupo` | Subgrupo técnico do procedimento |
| `co_forma_organizacao` | Forma de organização |
| Campos complementares do layout oficial | Complexidade, financiamento, valores, CBO, CID, compatibilidades e regras |


## 10. Salvamento da base tratada


In [ ]:
if not df_procedimentos.empty:
    arquivo_saida = DIRETORIO_TRATADO / "sigtap_procedimentos_minimo.parquet"
    df_procedimentos.to_parquet(arquivo_saida, index=False)
    print(f"Arquivo salvo em: {arquivo_saida}")
else:
    print("Base de procedimentos vazia. Verifique o ZIP do SIGTAP.")


## 11. Perguntas que conseguimos responder com SIGTAP

Com o SIGTAP, conseguimos responder perguntas semânticas sobre os procedimentos:

1. Qual é o nome do procedimento mais frequente no SIH/SUS?
2. Quais grupos de procedimento mais pressionam a rede hospitalar?
3. O crescimento de demanda está em procedimentos clínicos, cirúrgicos ou diagnósticos?
4. Quais procedimentos ambulatoriais especializados mais cresceram no SIA/SUS?
5. Quais procedimentos de média e alta complexidade aparecem com maior volume?
6. Quais procedimentos têm maior valor aprovado no SIH/SUS ou SIA/SUS?
7. Quais procedimentos estão associados a maior permanência hospitalar?
8. Quais grupos de atendimento devem ser destacados no dashboard executivo?
9. Como traduzir códigos técnicos para perguntas em linguagem natural?
10. Quais nomes de procedimentos devem ser usados no Select AI para melhorar a compreensão do gestor?


## 12. Exemplo — Enriquecimento de uma base SIH/SUS ou SIA/SUS

Abaixo está a lógica de join. Ela será usada quando as bases SIH/SUS e SIA/SUS já estiverem carregadas.


In [ ]:
# Exemplo didático de enriquecimento de procedimentos
exemplo_sih = pd.DataFrame({
    "PROC_REA": ["0303010037", "0407030026", "0202010503"],
    "internacoes": [120, 45, 300]
})

if not df_procedimentos.empty:
    exemplo_sih_enriquecido = exemplo_sih.merge(
        df_procedimentos,
        left_on="PROC_REA",
        right_on="co_procedimento",
        how="left"
    )
    display(exemplo_sih_enriquecido)
else:
    print("Carregue df_procedimentos para executar o enriquecimento.")


## 13. Como o SIGTAP melhora o Select AI

O Select AI tende a funcionar melhor quando os metadados são claros. Por isso, em vez de expor apenas `PROC_REA` ou `PA_PROC_ID`, o modelo analítico deve conter campos como:

- `codigo_procedimento`;
- `nome_procedimento`;
- `grupo_procedimento`;
- `subgrupo_procedimento`;
- `complexidade`;
- `tipo_financiamento`.

Exemplo de pergunta que fica mais natural:

```text
Quais grupos de procedimentos cirúrgicos tiveram maior crescimento de internações em 2024?
```

Em vez de:

```text
Agrupe por PROC_REA e calcule a variação mensal.
```
